**데이터 출처**: [Kaggle - Ad A/B Testing](https://www.kaggle.com/datasets/osuolaleemmanuel/ad-ab-testing) (osuolaleemmanuel)
**다운로드 날짜**: 2026-07-31


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("AdSmartABdata - AdSmartABdata.csv")

print(df.shape)
print(df.dtypes)
df.head()


(8077, 9)
auction_id       str
experiment       str
date             str
hour           int64
device_make      str
platform_os    int64
browser          str
yes            int64
no             int64
dtype: object


,auction_id,experiment,date,hour,device_make,platform_os,browser,yes,no
0,0008ef63-77a7-448b-bd1e-075f42c55e39,exposed,2020-07-10,8,Generic Smartphone,6,Chrome Mobile,0,0
1,000eabc5-17ce-4137-8efe-44734d914446,exposed,2020-07-07,10,Generic Smartphone,6,Chrome Mobile,0,0
2,0016d14a-ae18-4a02-a204-6ba53b52f2ed,exposed,2020-07-05,2,E5823,6,Chrome Mobile WebView,0,1
3,00187412-2932-4542-a8ef-3633901c98d9,control,2020-07-03,15,Samsung SM-A705FN,6,Facebook,0,0
4,001a7785-d3fe-4e11-a344-c8735acacc2c,control,2020-07-03,15,Generic Smartphone,6,Chrome Mobile,0,0


### 컬럼 설명

| 컬럼 | 의미 |
|---|---|
| `auction_id` | 유저(노출) 고유 ID |
| `experiment` | 처치 그룹 — `control`(더미/PSA 광고를 봄) vs `exposed`(실제 광고를 봄) |
| `date` | 노출 발생 날짜 |
| `hour` | 노출 발생 시간대(0~23) |
| `device_make` | 기기 제조사/모델 |
| `platform_os` | OS 플랫폼 코드 |
| `browser` | 사용 브라우저 |
| `yes` | BIO 서베이 질문("이 브랜드를 아시나요?")에 "예"라고 답했으면 1 |
| `no` | 같은 질문에 "아니오"라고 답했으면 1 (`yes`, `no` 둘 다 0이면 무응답) |


### 1. 데이터 점검

In [2]:
# --- 결측치 확인 ---
print(df.isnull().sum())


auction_id     0
experiment     0
date           0
hour           0
device_make    0
platform_os    0
browser        0
yes            0
no             0
dtype: int64


결측치 없음 — 9개 컬럼 모두 결측 0건

In [3]:
# --- 중복 확인 (auction_id가 유저/노출 고유 ID) ---
n_dup = df["auction_id"].duplicated().sum()
print(f"중복된 auction_id 수: {n_dup}")


중복된 auction_id 수: 0


`auction_id` 중복 0건 — 행 하나당 유저 하나, 중복 노출 없음

정제 전 8,077행 → 정제 후 8,077행 (제거된 행 없음)

In [4]:
# --- 배정 값 확인 (experiment가 control/exposed 외 다른 값이 없는지) ---
print(df["experiment"].value_counts())


experiment
control    4071
exposed    4006
Name: count, dtype: int64


`experiment`는 `control`(4,071명), `exposed`(4,006명) 두 값만 존재 — 배정 라벨에 오류나 이상값 없음

In [5]:
# --- SRM 점검: control vs exposed 그룹 크기가 의도한 비율(50:50)인가 ---
group_counts = df["experiment"].value_counts()
n_a, n_b = group_counts.values
chi2, p_srm = stats.chisquare([n_a, n_b], [(n_a + n_b) / 2] * 2)
print(f"그룹 크기: {dict(group_counts)}")
print(f"SRM 검정: chi2={chi2:.4f}, p={p_srm:.4f}  (p가 커야 정상)")

그룹 크기: {'control': np.int64(4071), 'exposed': np.int64(4006)}
SRM 검정: chi2=0.5231, p=0.4695  (p가 커야 정상)


SRM 검정: χ²=0.52, p=0.4695(>0.05) → 그룹 크기 비율(약 50:50)에 이상 없음, 배정 로직 정상 작동

In [6]:
# --- 무응답 비율 확인: yes=0, no=0인 행 (설문에 응답 안 함) ---
df["responded"] = (df["yes"] + df["no"]) > 0
print(df.groupby("experiment")["responded"].mean())

experiment
control    0.143945
exposed    0.164004
Name: responded, dtype: float64


응답률: control 14.4% vs exposed 16.4% — exposed 그룹이 약 2%p 더 높게 응답함.
광고 노출 자체가 설문 참여도를 높였을 가능성이 있어, "응답자 중 yes 비율"만으로 비교하면 selection bias가 낄 수 있음.
→ 이 비대칭을 위생 점검 이슈로 기록하고, 다음 단계(지표 정의)에서 전체 모수 대비 인지율로 볼지 응답자 대비로 볼지 결정 필요.

### 2. 질문과 가설

**질문**: 광고의 효과가 유의미하게 브랜드 인지도를 높이는 효과를 가져왔는가?

**지표 정의**: 인지율 = 전체 유저 대비 `yes` 응답 비율 (무응답은 인지 못한 것으로 간주).
처치군과 대조군 간 응답률 자체에 차이가 있어서, 응답자 중 yes 비율만으로 비교하면 selection bias가 낄 수 있기 때문에 이 방식을 택함.


- **H0**: 처치군(exposed)과 대조군(control) 간의 인지도 응답률의 차이가 없다.
- **H1**: 처치군과 대조군 간의 인지도 응답률에 차이가 있다.
- **유의수준**: α = 0.05
- **검정 방향**: 양측검정 (한쪽 방향만 확신할 수 없어 안전하게 양측으로 설정)


### 3. 검정 선택

In [7]:
# 카이제곱검정 기대빈도 가정 확인

from scipy.stats import chi2_contingency

contingency = pd.crosstab(df["experiment"], df["yes"])
chi2, p, dof, expected = chi2_contingency(contingency)

print(contingency)
print("기대빈도:")
print(expected)


yes            0    1
experiment           
control     3807  264
exposed     3698  308
기대빈도:
[[3782.69840287  288.30159713]
 [3722.30159713  283.69840287]]




**선택한 검정**: 카이제곱 독립성 검정

**선택 이유**: 확인하려는 지표가 비율(인지율)이고, 비교할 그룹이 control과 exposed 2개이기 때문에 카이제곱검정을 선택했다.
t-검정이 아닌 이유는 지표가 연속형이 아닌 비율형이기 때문이다.

**가정 확인**: 기대빈도가 모두 5 이상을 넘으므로(263.7~3782.7) 카이제곱검정의 가정을 만족한다.


### 4. 검정 수행

In [8]:
import numpy as np
from scipy.stats import norm

# 비율 계산
n_control = group_counts["control"]
n_exposed = group_counts["exposed"]
yes_control = contingency.loc["control", 1]
yes_exposed = contingency.loc["exposed", 1]

p_control = yes_control / n_control
p_exposed = yes_exposed / n_exposed

# ① 점추정 (차이)
diff = p_exposed - p_control

# ② 95% 신뢰구간 (두 비율 차이, Wald 근사)
se = np.sqrt(p_control*(1-p_control)/n_control + p_exposed*(1-p_exposed)/n_exposed)
z = norm.ppf(0.975)
ci_low, ci_high = diff - z*se, diff + z*se

# ③ 효과 크기: 절대 차이(%p), 상대 차이(배수), Cramér's V
abs_diff_pp = diff * 100
rel_diff = p_exposed / p_control
n_total = n_control + n_exposed
cramers_v = np.sqrt(chi2 / n_total)

print(f"control 인지율: {p_control:.4f} ({p_control*100:.2f}%)")
print(f"exposed 인지율: {p_exposed:.4f} ({p_exposed*100:.2f}%)")
print(f"① 점추정(차이): {abs_diff_pp:+.2f}%p")
print(f"② 95% CI: [{ci_low*100:.2f}%p, {ci_high*100:.2f}%p]")
print(f"③ 상대 차이: {rel_diff:.2f}배 ({(rel_diff-1)*100:+.1f}%)")
print(f"   Cramér's V: {cramers_v:.4f}")
print(f"카이제곱 검정: chi2={chi2:.4f}, p={p:.4f}")


control 인지율: 0.0648 (6.48%)
exposed 인지율: 0.0769 (7.69%)
① 점추정(차이): +1.20%p
② 95% CI: [0.08%p, 2.32%p]
③ 상대 차이: 1.19배 (+18.6%)
   Cramér's V: 0.0230
카이제곱 검정: chi2=4.2639, p=0.0389


### 4. 검정 수행 결과

exposed 그룹의 인지율이 control보다 높았다 (7.69% vs 6.48%).

- **절대 차이**: +1.20%p [95% CI: 0.08%p, 2.32%p]
- **상대 차이**: 1.19배 (control 대비 +18.6% 증가)
- **검정 결과**: χ²(1) = 4.26, p = .039
- **효과 크기**: Cramér's V = 0.023

절대 차이(1.20%p)만 보면 작아 보이지만, 상대 차이(+18.6%)로 보면 꽤 커 보인다 — 기준(control)이 6.48%로 원래 작았기 때문에 같은 차이가 상대적으로 크게 나타난 것이다.
통계적으로는 유의미하지만(p < .05), 효과 크기(Cramér's V = 0.023)는 매우 작아 실질적으로 크다고 보기는 어렵다.
광고 비용과 비교해 도입 여부를 판단할 필요가 있다.


### 5. 한 걸음 더 - 비용까지 고려했을 때 유의미한가?

In [9]:
# 퍼널 역산 기반 인지 1건의 가치 (3단계 시나리오)
funnel_scenarios = {
    "보수적": {"구매전환율": 0.01, "평균구매액": 30000, "마진율": 0.20},
    "중간":   {"구매전환율": 0.02, "평균구매액": 60000, "마진율": 0.30},
    "낙관적": {"구매전환율": 0.05, "평균구매액": 120000, "마진율": 0.40},
}
for s in funnel_scenarios.values():
    s["인지1건가치"] = s["구매전환율"] * s["평균구매액"] * s["마진율"]

# 인지율 lift의 통계적 불확실성 (95% CI)
lift_scenarios = {"CI 하한": ci_low, "점추정": diff, "CI 상한": ci_high}

노출단가 = 100  # 원, 가정: 노출 1건당 광고 비용

rows = []
for f_name, f in funnel_scenarios.items():
    for l_name, lift in lift_scenarios.items():
        증분가치 = lift * f["인지1건가치"]
        rows.append({
            "퍼널가정": f_name,
            "인지1건가치(원)": f["인지1건가치"],
            "lift 시나리오": l_name,
            "노출당증분가치(원)": round(증분가치, 2),
            "ROI": round(증분가치 / 노출단가, 3),
        })

roi_table = pd.DataFrame(rows)
print(roi_table.to_string(index=False))

breakeven_value = 노출단가 / diff
print(f"\n손익분기 인지 1건 가치 (점추정 lift 기준): {breakeven_value:,.0f}원")


퍼널가정  인지1건가치(원) lift 시나리오  노출당증분가치(원)   ROI
 보수적       60.0     CI 하한        0.05 0.001
 보수적       60.0       점추정        0.72 0.007
 보수적       60.0     CI 상한        1.39 0.014
  중간      360.0     CI 하한        0.30 0.003
  중간      360.0       점추정        4.33 0.043
  중간      360.0     CI 상한        8.36 0.084
 낙관적     2400.0     CI 하한        2.02 0.020
 낙관적     2400.0       점추정       28.89 0.289
 낙관적     2400.0     CI 상한       55.75 0.557

손익분기 인지 1건 가치 (점추정 lift 기준): 8,309원


### 5. 한 걸음 더 - 비용까지 고려했을 때 유의미한가?

**분석 설계**
두 가지 불확실성을 축으로 민감도 분석을 진행했다.
1. **인지율 lift의 통계적 불확실성** — 점추정 +1.20%p, 95% CI [0.08%p, 2.32%p]
2. **퍼널 역산 기반 인지 1건의 가치** — 구매전환율 × 평균구매액 × 마진을 보수적/중간/낙관적 3단계로 가정

| 시나리오 | 구매전환율 | 평균구매액 | 마진 | 인지 1건 가치 |
|---|---|---|---|---|
| 보수적 | 1% | 30,000원 | 20% | 60원 |
| 중간 | 2% | 60,000원 | 30% | 360원 |
| 낙관적 | 5% | 120,000원 | 40% | 2,400원 |

**ROI 매트릭스 (노출단가 100원 기준)**

| 퍼널가정 | lift 시나리오 | 노출당증분가치 | ROI |
|---|---|---|---|
| 보수적 | CI 하한 | 0.05원 | 0.001 |
| 보수적 | 점추정 | 0.72원 | 0.007 |
| 보수적 | CI 상한 | 1.39원 | 0.014 |
| 중간 | CI 하한 | 0.30원 | 0.003 |
| 중간 | 점추정 | 4.33원 | 0.043 |
| 중간 | CI 상한 | 8.36원 | 0.084 |
| 낙관적 | CI 하한 | 2.02원 | 0.020 |
| 낙관적 | 점추정 | 28.89원 | 0.289 |
| 낙관적 | CI 상한 | 55.75원 | 0.557 |

**해석**

9개 시나리오 전부 ROI < 1로 손익분기 미달이다. 가장 낙관적인 조합(낙관적 퍼널 + lift CI 상한)조차 ROI 0.557로, 투입 비용의 55.7%밖에 회수하지 못한다.

손익분기 인지 1건 가치는 **8,309원**이다 — 노출단가 100원을 정당화하려면 인지 1건이 최소 8,309원의 가치를 만들어야 하는데, 시도한 3개 퍼널 시나리오(60원~2,400원)는 전부 이 문턱값에 한참 못 미친다.

이를 통해 인지율의 차이가 통계적으로는 유의미하지만, 해당 효과크기가 광고집행을 결정할 만큼 크지는 않다는 것을 알 수 있다.
- 통계 검정: 이 1.2%p 차이가 우연은 아니다 → 맞음 (p=.039)
- 비즈니스 판단: 이 1.2%p 차이가 노출당 100원을 정당화할 만큼 크냐 → 이 데이터·가정 하에서는 아니다

8,309원이라는 손익분기 가치는 생각보다 꽤 높은 편이다. 이 정도 가치가 나오려면 애초에 제품 단가 자체가 높아야 할 것 같은데, 예를 들면 전자제품 중에서도 이어폰이나 소형 가전 같은 저가 제품보다는 냉장고나 노트북처럼 단가가 큰 제품이어야 말이 된다. 반대로 저가·저관여 제품이라면 손익분기 자체를 넘기기 어려워 보이고, 그런 경우엔 이 광고를 계속할 이유가 약해진다.

In [10]:
# --- 로버스트니스 체크: 응답자 기준 인지율로 재검정 ---
responded_df = df[df["responded"]]
contingency_resp = pd.crosstab(responded_df["experiment"], responded_df["yes"])
chi2_resp, p_resp, dof_resp, expected_resp = chi2_contingency(contingency_resp)

p_control_resp = contingency_resp.loc["control", 1] / (responded_df["experiment"] == "control").sum()
p_exposed_resp = contingency_resp.loc["exposed", 1] / (responded_df["experiment"] == "exposed").sum()

print(contingency_resp)
print(f"control 인지율(응답자 기준): {p_control_resp:.4f} ({p_control_resp*100:.2f}%)")
print(f"exposed 인지율(응답자 기준): {p_exposed_resp:.4f} ({p_exposed_resp*100:.2f}%)")
print(f"카이제곱 검정: chi2={chi2_resp:.4f}, p={p_resp:.4f}")


yes           0    1
experiment          
control     322  264
exposed     349  308
control 인지율(응답자 기준): 0.4505 (45.05%)
exposed 인지율(응답자 기준): 0.4688 (46.88%)
카이제곱 검정: chi2=0.3465, p=0.5561


**로버스트니스 체크 해석**

응답자 기준으로 재검정한 결과, 절대 차이는 오히려 더 컸지만(+1.83%p vs +1.20%p) 통계적으로 유의하지 않았다(p=.556 vs p=.039).
이는 원래 결론이 응답자만 남기면서 표본이 8,077명 → 1,243명으로 줄어든 데 따른 결과일 수 있다 — 즉 "효과가 있다"는 결론 자체가 지표 정의(무응답 처리 방식)라는 분석적 선택 하나에 좌우되는, **견고하지 않은(non-robust) 결과**다.


### 6. 의사결정

1. **질문과 가설** — 실제 광고 노출(exposed)이 브랜드 인지도를 control 대비 높였는가? H0: 차이 없음 / H1: 차이 있음, α=.05, 양측검정

2. **데이터 위생 점검** — 결측치·중복 없음, SRM 정상(p=.4695). 단, 응답률이 그룹 간 다름(control 14.4% vs exposed 16.4%)을 확인 — 응답자 기준이 아닌 전체 유저 기준 인지율로 지표를 정의해 이 비대칭의 영향을 피함

3. **검정 선택 근거** — 비율형 지표 × 2집단 비교 → 카이제곱 독립성 검정. 기대빈도 모두 5 이상으로 가정 충족

4. **결과** — exposed 인지율 7.69% vs control 6.48% (+1.20%p, 95% CI [0.08%p, 2.32%p], 상대 차이 +18.6%), χ²(1)=4.26, p=.039, Cramér's V=0.023 → 통계적으로 유의하나 효과 크기는 매우 작음

5. **의사결정**: **추가 수집**
브랜드의 주요 제품 단가를 실제로 수집해서 다시 의사결정한다 (내부 매출 데이터 또는 실제 인지→구매 퍼널 트래킹을 통해 평균구매액·마진율 확보).
다만 로버스트니스 체크 결과, "광고가 인지도를 높였다"는 결론 자체가 지표 정의(전체 기준 vs 응답자 기준)에 따라 유의성이 사라질 만큼 견고하지 않다는 것이 드러났다. 따라서 추가 수집은 제품 단가뿐 아니라, **응답률을 높이거나 표본을 늘린 재실험을 통해 효과의 존재 여부 자체를 다시 확인하는 것**을 포함해야 한다. 두 조건(① 효과가 견고하게 재현되고 ② 실제 인지 가치가 8,309원을 넘음)을 모두 만족하지 않는 한 광고는 하지 않는 방향으로 결정한다.

6. **한계**
브랜드의 실제 도메인을 알지 못해 결론을 내리기 어렵다. 현재는 평균구매액을 3만원에서 최대 12만원으로 가정해 민감도 분석을 진행했으며, 이 가정 범위를 벗어나는 고관여·고단가 상품이라면 결론이 달라질 수 있다. 또한 노출단가(100원) 역시 실제 광고 비용을 검증하지 않은 가정값이라, 인지 1건의 가치뿐 아니라 비용 쪽에도 불확실성이 남아있다.
또한 무응답을 "인지 못함(no)"으로 처리하는 것 자체도 완벽한 선택은 아니다 — 이는 보수적 가정으로, 실제로는 무응답이 인지 여부와 무관하게 발생했을 수 있다(예: 단순히 설문 자체가 귀찮아서 응답하지 않은 경우). 반대로 응답자 기준으로 계산하면 앞서 확인한 응답률 차이로 인한 selection bias 위험이 있다. 즉 어느 지표 정의를 택하든 완전히 편향에서 자유롭지는 않다.

